# Geometric interpretations

In [1]:
import numpy as np
import joblib
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from tensorflow import keras


## Naive comparison

In [2]:
PROJECT_ROOT = Path.cwd().parent

seed = 42
L = 18


# load axes
base_dir = PROJECT_ROOT / "runs" / "dgp0" / f"seed_{seed}"

v_base = np.load(base_dir / "baseline" / "scoring" / f"v_hat_L{L}.npy")
v_beta = np.load(base_dir / "beta_only" / "scoring" / f"v_hat_L{L}.npy")
v_kappa = np.load(base_dir / "kappa_only" / "scoring" / f"v_hat_L{L}.npy")


In [3]:
fig = go.Figure()

# baseline
fig.add_trace(go.Scatter(
    x=[0, v_base[0]],
    y=[0, v_base[1]],
    mode="lines+markers",
    name="Baseline axis",
    line=dict(width=4, color="black")
))

# beta only
fig.add_trace(go.Scatter(
    x=[0, v_beta[0]],
    y=[0, v_beta[1]],
    mode="lines+markers",
    name="Beta-only axis",
    line=dict(width=4, color="blue")
))

# kappa only
fig.add_trace(go.Scatter(
    x=[0, v_kappa[0]],
    y=[0, v_kappa[1]],
    mode="lines+markers",
    name="Kappa-only axis",
    line=dict(width=4, color="red")
))

fig.update_layout(
    title="Normalized Conduct Axes (Visual Comparison Only)",
    xaxis_title="Latent dimension 1",
    yaxis_title="Latent dimension 2",
    template="plotly_white",
    width=700,
    height=450
)

# keep aspect ratio equal
fig.update_yaxes(scaleanchor="x", scaleratio=1)

fig.show()

# Same AE comparison 

In [4]:
FEATURES_5 = ["volatility", "zero_change_fraction", "max_abs_ret", "AR_1", "price_range"]

baseline_dir = base_dir / "baseline"

#loading scaler and model
scaler = joblib.load(baseline_dir / "model" / f"scaler_L{L}.pkl")
encoder = keras.models.load_model(baseline_dir / "model" / f"encoder_L{L}.keras")


In [5]:
#helper functions
def compute_axis_in_baseline_space(mode: str):
    feat_path = base_dir / mode / "data" / "features" / f"features_L{L}.parquet"
    df = pd.read_parquet(feat_path).dropna(subset=FEATURES_5).copy()

    # project into BASELINE latent space
    X = df[FEATURES_5].to_numpy().astype(np.float32)
    Xs = scaler.transform(X).astype(np.float32)
    Z = encoder.predict(Xs, batch_size=2048, verbose=0)

    df["z1"] = Z[:, 0]
    df["z2"] = Z[:, 1]

    # centroids using pure windows only
    pure = df[df["is_pure_80"] == 1].copy()

    mu_C = pure[pure["state_mode"] == 0][["z1", "z2"]].mean().to_numpy()
    mu_K = pure[pure["state_mode"] == 2][["z1", "z2"]].mean().to_numpy()

    v = mu_K - mu_C
    v_hat = v / (np.linalg.norm(v) + 1e-12)

    return {
        "df": df,
        "pure": pure,
        "mu_C": mu_C,
        "mu_K": mu_K,
        "v_hat": v_hat
    }


In [6]:
res_base  = compute_axis_in_baseline_space("baseline")
res_beta  = compute_axis_in_baseline_space("beta_only")
res_kappa = compute_axis_in_baseline_space("kappa_only")

In [7]:
#plot helper
def add_axis(res, name, color):
    mu_C = res["mu_C"]
    mu_K = res["mu_K"]

    # centroid points
    fig.add_trace(go.Scatter(
        x=[mu_C[0], mu_K[0]],
        y=[mu_C[1], mu_K[1]],
        mode="markers",
        marker=dict(size=9, color=color),
        name=f"{name} centroids"
    ))

    # arrow/line
    fig.add_trace(go.Scatter(
        x=[mu_C[0], mu_K[0]],
        y=[mu_C[1], mu_K[1]],
        mode="lines",
        line=dict(width=4, color=color),
        name=f"{name} axis"
    ))

In [11]:
#plots
fig = go.Figure()


add_axis(res_base,  "Baseline",  "black")
add_axis(res_beta,  "Beta-only", "blue")
add_axis(res_kappa, "Kappa-only","red")

fig.update_layout(
    title="Conduct Axes Recomputed in Baseline Latent Space",
    xaxis_title="Latent dimension 1",
    yaxis_title="Latent dimension 2",
    template="plotly_white",
    width=800,
    height=500
)

fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

## Cosine simlarity 

In [12]:
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("cos(base, beta) =", cos_sim(res_base["v_hat"], res_beta["v_hat"]))
print("cos(base, kappa) =", cos_sim(res_base["v_hat"], res_kappa["v_hat"]))

cos(base, beta) = 0.9569671
cos(base, kappa) = 0.9953412


## Spread

In [13]:
v_base = res_base["v_hat"]

def projected_sep(res, axis):
    mu_C = res["mu_C"]
    mu_K = res["mu_K"]
    return float(np.dot(mu_K - mu_C, axis))

d_base = projected_sep(res_base, v_base)
d_beta = projected_sep(res_beta, v_base)
d_kappa = projected_sep(res_kappa, v_base)

print("Projected separation on baseline axis")
print("baseline :", d_base)
print("beta_only:", d_beta)
print("kappa_only:", d_kappa)

print("\nSpread ratios relative to baseline")
print("beta_only :", d_beta / d_base)
print("kappa_only:", d_kappa / d_base)

Projected separation on baseline axis
baseline : 2.0677576065063477
beta_only: 1.2977497577667236
kappa_only: 1.2307558059692383

Spread ratios relative to baseline
beta_only : 0.6276121309786316
kappa_only: 0.5952128054548448
